# Matemática III — Primer Trabajo Práctico

**Estimación de los niveles de obesidad a partir de los hábitos alimentarios y la condición física**
**Alumna: Camila Maldonado**

## Introducción
Este trabajo analiza el conjunto de datos *Estimation of Obesity Levels Based on Eating Habits and Physical Condition*. Reúne 2.111 registros de personas de México, Perú y Colombia y 17 variables vinculadas con características personales, alimentación, actividad física y uso de tecnología. Según la fuente, combina respuestas de encuestas web con registros generados sintéticamente; por ello, los resultados describen este conjunto y no deben interpretarse como evidencia clínica ni causal.

La pregunta que guía el análisis es: **¿en qué medida las características personales y los hábitos permiten estimar el peso y clasificar el nivel de obesidad registrado?**.

Para la clasificación, la variable objetivo es `NIV_OB` (originalmente `NObeyesdad`), con siete categorías: **Peso insuficiente, Peso normal, Sobrepeso nivel I, Sobrepeso nivel II, Obesidad tipo I, Obesidad tipo II y Obesidad tipo III**.

Para facilitar la lectura y el análisis del modelo, se traducieron todas las variables categóricas al español. Por otro lado, se calculara el IMC para enriquecer la interpretación de los datos, pero no lo utilizaremos como variable predictora, ya que al derivarse directamente del peso y la altura podría introducir circularidad en el análisis.


**Descripción de las variables**
- `GENERO`: género reportado.
- `EDAD`: edad en años.
- `ALTURA`: altura en metros.
- `PESO`: peso en kilogramos.
- `HFS`: antecedentes familiares de sobrepeso.
- `CCF`: consumo frecuente de alimentos calóricos.
- `CVE`: frecuencia de consumo de vegetales.
- `CPD`: número de comidas principales.
- `CEC`: consumo entre comidas.
- `FUMA`: consumo de tabaco.
- `CH2O`: consumo de agua según escala registrada.
- `CTRL_CALC`: control de calorías.
- `AFF`: frecuencia de actividad física.
- `TDP`: tiempo de uso de dispositivos.
- `CALC`: consumo de alcohol.
- `MTRANS`: medio de transporte habitual.
- `NIV_OB`: nivel de obesidad registrado y variable objetivo de clasificación.

Fuente: [UCI Machine Learning Repository — stimation of Obesity Levels Based on Eating Habits and Physical Condition](https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition).

## 1. Importación de librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
    balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 50)
print('Librerías importadas. Semilla reproducible:', RANDOM_STATE)

## 2. Carga del dataset
Colocá `ObesityDataSet.csv` en la misma carpeta que este notebook. El bloque también busca el archivo en la carpeta de trabajo de Google Colab.

In [ ]:
rutas = [Path('ObesityDataSet.csv'), Path('/content/ObesityDataSet.csv')]
ruta_csv = next((ruta for ruta in rutas if ruta.exists()), None)
if ruta_csv is None:
    raise FileNotFoundError('No se encontró ObesityDataSet.csv. Colocalo junto al notebook o en /content/.')
df = pd.read_csv(ruta_csv)

# Diccionario de traducción a siglas en español
siglas_espanol = {
    'Gender': 'GENERO',
    'Age': 'EDAD',
    'Height': 'ALTURA',
    'Weight': 'PESO',
    'family_history_with_overweight': 'HFS',
    'FAVC': 'CCF',
    'FCVC': 'CVE',
    'NCP': 'CPD',
    'CAEC': 'CEC',
    'SMOKE': 'FUMA',
    'CH2O': 'CH2O',
    'SCC': 'CTRL_CALC',
    'FAF': 'AFF',
    'TUE': 'TDP',
    'CALC': 'CALC',
    'MTRANS': 'MTRANS',
    'NObeyesdad': 'NIV_OB'
}

# Aplicar los cambios al DataFrame
df.rename(columns=siglas_espanol, inplace=True)

# Traducir todas las categorías de texto al español
traducciones = {
    'GENERO': {'Male': 'Varón', 'Female': 'Mujer'},
    'HFS': {'yes': 'Sí', 'no': 'No'},
    'CCF': {'yes': 'Sí', 'no': 'No'},
    'CEC': {'Sometimes': 'A veces', 'Frequently': 'Frecuentemente', 'Always': 'Siempre', 'no': 'No consume'},
    'FUMA': {'yes': 'Sí', 'no': 'No'},
    'CTRL_CALC': {'yes': 'Sí', 'no': 'No'},
    'CALC': {'Sometimes': 'A veces', 'Frequently': 'Frecuentemente', 'Always': 'Siempre', 'no': 'No consume'},
    'MTRANS': {'Public_Transportation': 'Transporte público', 'Automobile': 'Automóvil', 'Walking': 'A pie', 'Motorbike': 'Motocicleta', 'Bike': 'Bicicleta'},
    'NIV_OB': {'Insufficient_Weight': 'Peso insuficiente', 'Normal_Weight': 'Peso normal', 'Overweight_Level_I': 'Sobrepeso nivel I', 'Overweight_Level_II': 'Sobrepeso nivel II', 'Obesity_Type_I': 'Obesidad tipo I', 'Obesity_Type_II': 'Obesidad tipo II', 'Obesity_Type_III': 'Obesidad tipo III'}
}
for columna, mapa in traducciones.items():
    if columna in df.columns:
        df[columna] = df[columna].replace(mapa)

descripcion = pd.DataFrame({
    'Sigla': ['GENERO','EDAD','ALTURA','PESO','HFS','CCF','CVE','CPD','CEC','FUMA','CH2O','CTRL_CALC','AFF','TDP','CALC','MTRANS','NIV_OB'],
    'Descripción en español': ['Género reportado','Edad en años','Altura en metros','Peso en kilogramos','Antecedentes familiares de sobrepeso','Consumo frecuente de alimentos calóricos','Frecuencia de consumo de vegetales','Número de comidas principales','Consumo entre comidas','Consumo de tabaco','Consumo de agua según escala registrada','Control de calorías','Frecuencia de actividad física','Tiempo de uso de dispositivos','Consumo de alcohol','Medio de transporte habitual','Nivel de obesidad registrado (objetivo)']
})
display(df.head())

## 3. Análisis Exploratorio (EDA)
### 3.1 Información general

In [ ]:
print('=== Dimensiones del dataset ===')
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print('\n=== Tipos de datos ===')
display(df.dtypes.rename('Tipo de dato').to_frame())
print('\n=== Valores nulos por columna ===')
display(df.isnull().sum().rename('Valores nulos').to_frame())
print(f'\nFilas duplicadas: {df.duplicated().sum()}')
print('\n=== Resumen descriptivo ===')
display(df.describe(include='all').T)

> **Observación:** El conjunto contiene 2.111 registros y 17 variables. No se observan valores nulos, por lo que no es necesario imputar ni eliminar filas por datos faltantes. Se detectan 24 filas duplicadas; conviene reconocerlo como una limitación y, para conservar la comparabilidad con los resultados presentados, este análisis mantiene los registros originales.

### 3.2 Distribución de `NIV_OB`

In [ ]:
orden = df['NIV_OB'].value_counts().index
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='NIV_OB', order=orden)
plt.title('Distribución de niveles de obesidad registrados')
plt.xlabel('Nivel registrado'); plt.ylabel('Cantidad de observaciones')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
display((df['NIV_OB'].value_counts(normalize=True)*100).round(1).rename('Porcentaje').to_frame())

> **Observación:** El gráfico muestra que las siete categorías están relativamente balanceadas: sus proporciones se ubican aproximadamente entre el 12,9 % y el 16,6 %. *Obesidad tipo I* es la categoría más frecuente (351 personas; 16,6 %) y *Peso insuficiente* la menos frecuente (272; 12,9 %).

### 3.3 Distribuciones de variables numéricas

In [ ]:
numericas = df.select_dtypes(include=np.number).columns.tolist()
df[numericas].hist(figsize=(14,10), bins=25, edgecolor='white')
plt.suptitle('Distribuciones de variables numéricas', y=1.02)
plt.tight_layout(); plt.show()

> **Observación:** Las variables numéricas presentan rangos y escalas diferentes; por eso, los modelos se ajustarán con estandarización dentro de un flujo de trabajo. Algunas variables numéricas (por ejemplo, frecuencia de actividad física o consumo de agua) representan escalas de respuesta, no conteos absolutos.

### 3.4 Matriz de correlación

In [ ]:
fig, ax = plt.subplots(figsize=(11,7))
sns.heatmap(df[numericas].corr(), annot=True, cmap='vlag', center=0, fmt='.2f', ax=ax)
ax.set_title('Matriz de correlación de variables numéricas')
plt.tight_layout(); plt.show()

> **Observación:** Los coeficientes próximos a 1 o −1 reflejan asociaciones lineales más intensas, mientras que los próximos a cero indican asociaciones lineales débiles. La correlación ayuda a identificar relaciones y posibles redundancias, pero no demuestra causalidad ni garantiza que una relación sea útil para predecir.

### 3.5 IMC según nivel de obesidad

In [ ]:
df['IMC'] = df['PESO'] / df['ALTURA']**2
plt.figure(figsize=(11,5))
sns.boxplot(data=df, x='NIV_OB', y='IMC', order=orden)
plt.title('IMC calculado por nivel de obesidad registrado')
plt.xlabel('Nivel registrado'); plt.ylabel('IMC (kg/m²)')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()
resumen_imc = df.groupby('NIV_OB')['IMC'].agg(['count','median','mean','min','max']).reindex(orden).round(2)
display(resumen_imc)

> **Observación:** El diagrama de cajas compara la distribución del IMC entre los niveles registrados; la línea central representa la mediana y la caja el rango intercuartílico. Si las medianas crecen hacia las categorías de obesidad, se observa una asociación coherente entre ambas medidas. Esto no implica que el IMC cause la etiqueta, y el IMC no es un diagnóstico individual.

### 3.6 Comparación descriptiva entre mujeres y varones
Se compararán los porcentajes de `NIV_OB` dentro de cada género para que los tamaños distintos de los grupos no distorsionen la lectura. También se observarán las distribuciones del IMC.


In [ ]:
print('Composición de la muestra por género (%)')
display((df['GENERO'].value_counts(normalize=True)*100).round(1).rename('Porcentaje').to_frame())
plt.figure(figsize=(11,5))
porc = pd.crosstab(df['GENERO'], df['NIV_OB'], normalize='index')*100
porc.plot(kind='bar', stacked=True, figsize=(11,5), colormap='Set2')
plt.title('Distribución porcentual de niveles registrados dentro de cada género')
plt.xlabel('Género reportado'); plt.ylabel('Porcentaje dentro del género')
plt.legend(title='Nivel registrado', bbox_to_anchor=(1.02,1), loc='upper left')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
display(porc.round(1))
fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(data=df, x='GENERO', y='IMC', ax=ax)
ax.set_title('Distribución del IMC según género reportado'); ax.set_xlabel('Género reportado'); ax.set_ylabel('IMC (kg/m²)')
plt.tight_layout(); plt.show()
display(df.groupby('GENERO')['IMC'].median().rename('Mediana IMC (kg/m²)').to_frame().round(2))

> ***Observación:** El gráfico de barras apiladas compara la proporción de cada nivel de `NIV_OB` dentro de mujeres y varones, y el diagrama de cajas compara sus distribuciones de IMC. La tabla permite precisar qué categorías presentan mayores diferencias porcentuales en esta muestra. Estas son diferencias descriptivas; no permiten afirmar que el género las cause, ya que pueden estar asociadas con edad, hábitos, actividad física u otros factores. Los resultados corresponden a las categorías de género registradas en el dataset.*

### 3.7 Hábitos y niveles registrados

In [ ]:
for variable in ['HFS','CCF','CEC','FUMA','CTRL_CALC','CALC','MTRANS']:
    print(f'\nDistribución de NIV_OB según {variable} (%)')
    display((pd.crosstab(df[variable], df['NIV_OB'], normalize='index')*100).round(1))

> **Observación:** Las distribuciones porcentuales varían entre algunos grupos de hábitos y transporte, lo que sugiere que estas variables podrían aportar información predictiva. Sin embargo, los patrones no prueban que un hábito cause un determinado nivel de obesidad: también pueden intervenir factores no medidos, sesgos de encuesta y la composición de la muestra.

## 4. Preparación de datos para los modelos
Se estimará `PESO` con regresiones lineales y se clasificará `NIV_OB` con regresión logística. Las transformaciones se ajustan dentro de un flujo de trabajo para prevenir filtraciones de información entre entrenamiento y prueba.

### 4.1. Criterios de preparación

En esta parte se trabajará con dos objetivos diferentes:

- **Regresión:** estimar el peso (`PESO`) a partir de las demás variables disponibles, sin utilizar el nivel de obesidad (`NIV_OB`) ni el IMC calculado.
- **Clasificación:** predecir el nivel de obesidad (`NIV_OB`) utilizando características y hábitos, sin incluir peso ni altura. Así se evita que el modelo utilice medidas corporales directamente relacionadas con la construcción de la categoría que intenta predecir.

Se separan los datos en entrenamiento (80 %) y prueba (20 %). El conjunto de prueba queda reservado para evaluar los modelos con datos que no participaron en el ajuste.

In [ ]:
# REGRESIÓN: el objetivo es estimar el peso.
X_reg = df.drop(columns=['PESO', 'NIV_OB', 'IMC'])
y_reg = df['PESO']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

cat_reg = X_reg.select_dtypes(exclude=np.number).columns.tolist()
num_reg = X_reg.select_dtypes(include=np.number).columns.tolist()

prep_reg = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), num_reg),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_reg)
])

print('Variables numéricas:', num_reg)
print('Variables categóricas:', cat_reg)
print(f'Entrenamiento: {len(Xr_train)} registros')
print(f'Prueba: {len(Xr_test)} registros')


**Observación:** se utiliza el 80 % de los registros para entrenar y el 20 % para evaluar. Las variables categóricas se convierten a formato numérico mediante codificación y las numéricas se estandarizan. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, evitando que la información del conjunto de prueba intervenga en el aprendizaje.


## 5. Regresión lineal simple

Se utiliza la altura como única variable explicativa para estimar el peso. Este modelo permite estudiar si existe una relación lineal entre ambas variables, aunque no contempla otros factores que también pueden estar relacionados con el peso.


In [ ]:
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    df[['ALTURA']], y_reg, test_size=0.20, random_state=RANDOM_STATE
)

modelo_simple = LinearRegression()
modelo_simple.fit(Xs_train, ys_train)
pred_simple = modelo_simple.predict(Xs_test)

mae_simple = mean_absolute_error(ys_test, pred_simple)
rmse_simple = np.sqrt(mean_squared_error(ys_test, pred_simple))
r2_simple = r2_score(ys_test, pred_simple)

print(f'Peso estimado = {modelo_simple.intercept_:.2f} + '
      f'{modelo_simple.coef_[0]:.2f} × altura')
print(f'MAE: {mae_simple:.2f} kg')
print(f'RMSE: {rmse_simple:.2f} kg')
print(f'R²: {r2_simple:.3f}')

plt.figure(figsize=(8, 5))
plt.scatter(Xs_test['ALTURA'], ys_test, alpha=0.55, label='Peso observado')
x_line = np.linspace(Xs_test['ALTURA'].min(), Xs_test['ALTURA'].max(), 200)
plt.plot(
    x_line,
    modelo_simple.predict(pd.DataFrame({'ALTURA': x_line})),
    color='red', label='Recta estimada'
)
plt.title('Regresión lineal simple: altura y peso')
plt.xlabel('Altura (m)')
plt.ylabel('Peso (kg)')
plt.legend()
plt.tight_layout()
plt.show()


**Observación:** los puntos representan los pesos observados y la recta muestra los valores estimados por el modelo a partir de la altura. Se observa una tendencia general positiva: a mayor altura, el peso estimado tiende a ser mayor. Sin embargo, los puntos presentan una separación considerable respecto de la recta. Esto indica que la altura por sí sola no permite estimar el peso con gran precisión. El valor de R² y los errores MAE y RMSE permiten cuantificar esta limitación.


### 5.1. Análisis de residuos de la regresión simple

Un residuo es la diferencia entre el valor real y el valor estimado por el modelo. En un gráfico de residuos no se busca que los puntos estén pegados a la línea horizontal: **es esperable que estén dispersos alrededor de cero**. Lo importante es que no formen patrones claros, como curvas o un embudo, que podrían indicar que el modelo lineal no representa adecuadamente la relación o que la variabilidad de los errores cambia.


In [ ]:
residuos_simple = ys_test.to_numpy() - pred_simple

plt.figure(figsize=(8, 5))
plt.scatter(pred_simple, residuos_simple, alpha=0.55)
plt.axhline(0, color='red', linestyle='--', label='Residuo cero')
plt.title('Gráfico de residuos: regresión lineal simple')
plt.xlabel('Peso estimado (kg)')
plt.ylabel('Residuo (peso real − estimado, kg)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Error medio de los residuos: {np.mean(residuos_simple):.2f} kg')
print(f'Desvío estándar de los residuos: {np.std(residuos_simple):.2f} kg')


**Observación:** los residuos se distribuyen a ambos lados de la línea cero y presentan una dispersión amplia. La dispersión alrededor de cero, por sí sola, no es un error: es el comportamiento que se espera de un gráfico de residuos. En este caso, la amplitud de los errores muestra que las estimaciones individuales pueden alejarse bastante del peso real. Esto coincide con el R² bajo de la regresión simple. Por lo tanto, aunque la altura aporta información, no alcanza para explicar gran parte de las diferencias de peso entre las personas.


## 6. Regresión lineal múltiple

A diferencia de la regresión simple, este modelo incorpora varias características de las personas, como edad, altura, hábitos alimentarios y actividad física. El objetivo es comprobar si el uso conjunto de estas variables mejora la estimación del peso.


In [ ]:
def metricas_regresion(y_real, y_pred, nombre):
    return {
        'Modelo': nombre,
        'MAE (kg)': mean_absolute_error(y_real, y_pred),
        'RMSE (kg)': np.sqrt(mean_squared_error(y_real, y_pred)),
        'R²': r2_score(y_real, y_pred)
    }

modelo_multiple = Pipeline([
    ('prep', prep_reg),
    ('reg', LinearRegression())
])
modelo_multiple.fit(Xr_train, yr_train)
pred_multiple = modelo_multiple.predict(Xr_test)

resultados_reg = [
    metricas_regresion(ys_test, pred_simple, 'Simple: altura'),
    metricas_regresion(yr_test, pred_multiple, 'Múltiple')
]
tabla_reg = pd.DataFrame(resultados_reg)
display(tabla_reg.round(3))

residuos_multiple = yr_test.to_numpy() - pred_multiple
plt.figure(figsize=(8, 5))
plt.scatter(pred_multiple, residuos_multiple, alpha=0.55)
plt.axhline(0, color='red', linestyle='--', label='Residuo cero')
plt.title('Gráfico de residuos: regresión lineal múltiple')
plt.xlabel('Peso estimado (kg)')
plt.ylabel('Residuo (peso real − estimado, kg)')
plt.legend()
plt.tight_layout()
plt.show()


**Observación:** al incorporar más variables explicativas, el modelo múltiple mejora el ajuste respecto de la regresión simple. Esto se refleja en una disminución de los errores y un aumento del R². En el gráfico de residuos se analiza si los errores quedan distribuidos alrededor de cero y si aparece algún patrón sistemático. Aunque una dispersión visible es normal, los errores todavía representan diferencias entre los valores reales y estimados. Por eso, el modelo mejora la estimación, pero no elimina toda la incertidumbre.


## 7. Regularización: Ridge y Lasso

Ridge y Lasso son métodos que agregan una penalización a los coeficientes de la regresión. Se utilizan para limitar coeficientes demasiado grandes y pueden ayudar cuando existen variables relacionadas entre sí. Se comparan con la regresión múltiple mediante las mismas métricas y el mismo conjunto de prueba.


In [ ]:
alphas = np.logspace(-3, 3, 25)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

modelo_ridge = Pipeline([
    ('prep', prep_reg),
    ('reg', RidgeCV(alphas=alphas, cv=cv_reg))
])
modelo_lasso = Pipeline([
    ('prep', prep_reg),
    ('reg', LassoCV(
        alphas=alphas, cv=cv_reg, max_iter=30000,
        random_state=RANDOM_STATE
    ))
])

modelo_ridge.fit(Xr_train, yr_train)
modelo_lasso.fit(Xr_train, yr_train)

pred_ridge = modelo_ridge.predict(Xr_test)
pred_lasso = modelo_lasso.predict(Xr_test)

resultados_reg += [
    metricas_regresion(yr_test, pred_ridge, 'Ridge'),
    metricas_regresion(yr_test, pred_lasso, 'Lasso')
]
tabla_reg = pd.DataFrame(resultados_reg)
display(tabla_reg.round(3))

print('Alpha seleccionado para Ridge:', modelo_ridge.named_steps['reg'].alpha_)
print('Alpha seleccionado para Lasso:', modelo_lasso.named_steps['reg'].alpha_)
print('Coeficientes distintos de cero en Lasso:',
      np.sum(modelo_lasso.named_steps['reg'].coef_ != 0))


**Observación:** la tabla permite comparar los cuatro modelos. Un MAE y un RMSE menores indican errores de estimación más bajos, mientras que un R² mayor indica que el modelo explica una mayor proporción de la variabilidad del peso. En estos datos, Ridge y Lasso presentan resultados muy cercanos a la regresión múltiple. Por lo tanto, la regularización no produce una mejora grande, aunque permite controlar los coeficientes del modelo.


## 8. Regresión logística: clasificación del nivel de obesidad

En esta sección se busca clasificar a las personas según el nivel de obesidad registrado. Para que la evaluación no dependa directamente de las medidas corporales relacionadas con la definición de las categorías, se excluyen `PESO` y `ALTURA`. Se utilizan las variables de hábitos y características restantes.

Se evalúa el modelo con exactitud, exactitud balanceada, F1 macro y matriz de confusión. El F1 macro considera el rendimiento de todas las clases por igual, por lo que resulta útil cuando interesa evaluar cada categoría y no solamente el resultado global.


In [ ]:
def construir_clasificador(X):
    cat = X.select_dtypes(exclude=np.number).columns.tolist()
    num = X.select_dtypes(include=np.number).columns.tolist()

    preparacion = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), num),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat)
    ])

    return Pipeline([
        ('preparacion', preparacion),
        ('clasificador', LogisticRegression(
            max_iter=3000, class_weight='balanced', C=1.0
        ))
    ])

y_clf = df['NIV_OB']
X_clf = df.drop(columns=['NIV_OB', 'PESO', 'ALTURA', 'IMC'])

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_STATE,
    stratify=y_clf
)

modelo_log = construir_clasificador(X_clf)
modelo_log.fit(Xc_train, yc_train)
pred_clf = modelo_log.predict(Xc_test)

print(f'Exactitud: {accuracy_score(yc_test, pred_clf):.3f}')
print(f'Exactitud balanceada: {balanced_accuracy_score(yc_test, pred_clf):.3f}')
print(f'F1 macro: {f1_score(yc_test, pred_clf, average="macro"):.3f}')

print('\nReporte de clasificación por categoría:')
print(classification_report(yc_test, pred_clf, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    yc_test, pred_clf, xticks_rotation=45,
    cmap='Blues', values_format='d'
)
plt.title('Matriz de confusión — regresión logística')
plt.tight_layout()
plt.show()


**Observación:** las métricas resumen el desempeño general, mientras que el reporte por clase permite observar qué categorías se reconocen mejor y cuáles presentan más confusiones. En la matriz de confusión, los aciertos se ubican en la diagonal principal y los errores fuera de ella. Como no se incluyen peso ni altura, el modelo debe reconocer patrones a partir de hábitos y otras características, lo cual representa una tarea más exigente. Los resultados no significan que los hábitos sean causas directas del nivel de obesidad.


## 9. Validación cruzada

La validación cruzada divide los datos en cinco partes o pliegues. En cada repetición se entrena el modelo con cuatro partes y se evalúa con la restante. De esta manera, se obtiene una estimación del rendimiento menos dependiente de una única división entre entrenamiento y prueba.

Se informa el promedio y el desvío estándar de las métricas. Un desvío estándar menor indica que el resultado varía menos entre los pliegues.


In [ ]:
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores_clf = cross_validate(
    modelo_log, X_clf, y_clf,
    cv=cv_clf,
    scoring={
        'f1_macro': 'f1_macro',
        'balanced_accuracy': 'balanced_accuracy'
    },
    n_jobs=-1
)

print('CLASIFICACIÓN')
print('F1 macro: {:.3f} ± {:.3f}'.format(
    scores_clf['test_f1_macro'].mean(),
    scores_clf['test_f1_macro'].std()
))
print('Exactitud balanceada: {:.3f} ± {:.3f}'.format(
    scores_clf['test_balanced_accuracy'].mean(),
    scores_clf['test_balanced_accuracy'].std()
))

cv_regresion = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
modelo_cv_reg = Pipeline([
    ('prep', prep_reg),
    ('reg', LinearRegression())
])

scores_reg = cross_validate(
    modelo_cv_reg, X_reg, y_reg,
    cv=cv_regresion,
    scoring={'r2': 'r2', 'neg_mae': 'neg_mean_absolute_error'},
    n_jobs=-1
)

print('\nREGRESIÓN MÚLTIPLE')
print('R²: {:.3f} ± {:.3f}'.format(
    scores_reg['test_r2'].mean(), scores_reg['test_r2'].std()
))
print('MAE: {:.2f} ± {:.2f} kg'.format(
    -scores_reg['test_neg_mae'].mean(),
    scores_reg['test_neg_mae'].std()
))


**Observación:** la validación cruzada permite comprobar si el rendimiento se mantiene al cambiar los datos utilizados para entrenar y evaluar. El promedio resume el resultado de los cinco pliegues y el desvío estándar muestra cuánto varía. Si la variabilidad es elevada, las métricas deben interpretarse con mayor cautela, ya que el desempeño puede depender de qué registros se utilicen para el entrenamiento.


## 10. Observaciones generales y conclusiones

El análisis exploratorio permitió describir las variables del conjunto de datos y observar diferencias entre los niveles de obesidad registrados. Las asociaciones encontradas son descriptivas: no permiten establecer que un hábito, característica o comportamiento sea la causa de un determinado nivel de obesidad.

En la regresión lineal simple se utilizó la altura para estimar el peso. El modelo mostró una capacidad explicativa limitada, lo que indica que una sola variable no alcanza para representar adecuadamente las diferencias de peso entre las personas. En el gráfico de residuos, la dispersión alrededor de cero es esperable; lo importante es evaluar si aparecen patrones sistemáticos. La amplitud de los residuos y el R² obtenido muestran que las estimaciones individuales todavía presentan errores considerables.

La regresión múltiple mejoró el desempeño al incorporar más características. Ridge y Lasso obtuvieron resultados similares a este modelo, por lo que la regularización no produjo una diferencia importante en esta ejecución. La comparación debe realizarse observando conjuntamente MAE, RMSE y R², ya que cada métrica describe un aspecto distinto del error y del ajuste.

En la clasificación mediante regresión logística se intentó predecir el nivel de obesidad sin utilizar directamente el peso ni la altura. El resultado permite evaluar cuánto aportan los hábitos y las demás características disponibles, aunque algunas categorías pueden confundirse entre sí. La matriz de confusión y las métricas por clase ayudan a identificar esas diferencias.

Por último, la validación cruzada aporta una evaluación complementaria de la estabilidad de los modelos. En conjunto, los resultados muestran que es posible encontrar relaciones y construir modelos predictivos con este dataset, pero sus estimaciones no son exactas ni permiten establecer causalidad. Además, el conjunto contiene registros observacionales y duplicados, por lo que las conclusiones deben limitarse a los datos analizados y no generalizarse automáticamente a toda la población.


In [ ]:
# Resumen numérico de los resultados obtenidos en esta ejecución
display(tabla_reg.round(3))

mejor_reg = tabla_reg.loc[tabla_reg['MAE (kg)'].idxmin()]
print(
    f"El menor MAE corresponde a {mejor_reg['Modelo']}: "
    f"{mejor_reg['MAE (kg)']:.2f} kg; "
    f"RMSE={mejor_reg['RMSE (kg)']:.2f} kg; "
    f"R²={mejor_reg['R²']:.3f}."
)
print(
    f"Regresión logística: exactitud={accuracy_score(yc_test, pred_clf):.3f}, "
    f"exactitud balanceada={balanced_accuracy_score(yc_test, pred_clf):.3f}, "
    f"F1 macro={f1_score(yc_test, pred_clf, average='macro'):.3f}."
)
